In [2]:
# import libraries 
import time
import re
import datetime
import csv
import logging
import sys
import os
import random

from selenium import webdriver
from bs4 import BeautifulSoup

# cần 2 cái dưới để điều hướng con trỏ chuột sang trang kế
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import logging # dùng để ghi tiến trình & lỗi khi crawl data

# Import 3 vũ khí mới để vượt qua bước Login của web
import os # để đọc thông tin trong file `.env`
import undetected_chromedriver as uc
from dotenv import load_dotenv

In [3]:
# --- Thiết lập logging ---
logging.basicConfig(
    filename="logging_careerviet.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    encoding='utf-8'
)

# 1. Truy cập web

In [4]:
# https://careerviet.vn/viec-lam/Data-Analyst-k-vi.html
# https://careerviet.vn/viec-lam/Data-Analyst-k-trang-4-vi.html
# https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-2-vi.html

KEYWORDS = ["Data Analyst"] # Rút gọn để test
BASE_URL_TEMPLATE = "https://careerviet.vn/viec-lam/{keyword}-k-sortdv-trang-{page}-vi.html"

MAX_PAGES = 3
OUTPUT_CSV = "data_careerviet.csv"

# ── 3. LOGGING (Giữ nguyên của bạn) ──────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")

In [5]:
# THAY ĐỔI 1: Dùng undetected_chromedriver thay vì webdriver.Chrome()
options = uc.ChromeOptions()


# Thêm cấu hình vô hiệu hóa chức năng hỏi lưu mật khẩu VÀ chặn thông báo (Notifications)
prefs = {
    "credentials_enable_service": False,
    "profile.password_manager_enabled": False,
    "profile.default_content_setting_values.notifications": 2  # Thêm dòng này! 2 có nghĩa là Block

}
options.add_experimental_option("prefs", prefs)



# THÊM version_main=140 VÀO ĐÂY vì nó mặc định chạy ver Chrome 147
driver = uc.Chrome(options=options, version_main=140) 


# Thay vì maximize_window(), dùng set_window_size và set_window_position
# Đặt kích thước cho trình duyệt (chiều rộng x chiều cao)
driver.set_window_size(900, 1080) # Bạn có thể thay đổi số theo độ phân giải màn hình
# (Tùy chọn) Đặt vị trí xuất hiện của cửa sổ (x, y) trên màn hình (0,0 là góc trên cùng bên trái)
driver.set_window_position(900, 0) # Có thể đẩy nó sang lề phải của màn hình

{'height': 882, 'width': 903, 'x': 900, 'y': 0}

# 2. Lấy link job trong 1 page

In [ ]:
# Mở trang TopCV (trang 1)
url = 'https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-2-vi.html'
driver.get(url)
time.sleep(random.uniform(3, 6)) # Đợi 3 giây để trang web tải hoàn toàn

In [32]:
# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

In [33]:
# Lấy mã nguồn HTML của trang web
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

In [ ]:
soup

In [ ]:
# 1. Tìm container chứa danh sách công việc BẰNG "jobs-side-list" (đã dùng find all và chỉ ra 1 kết quả duy nhất, trộm vía)
job_list_box = soup.find("div", class_ = "jobs-side-list", id="jobs-side-list-content")
job_list_box

<div class="jobs-side-list" id="jobs-side-list-content"><div class="job-item" id="job-item-35C71953"><div class="figure"><div class="asset-wrapper"><div class="image"><a href="/vi/nha-tuyen-dung/cong-ty-tnhh-san-xuat-thuong-mai-dich-vu-hoa-xinh.35A9E5F2.html" target="_blank" title="Công ty TNHH Sản xuất Thương mại &amp; Dịch vụ Hoa Xinh"><img alt="Công ty TNHH Sản xuất Thương mại &amp; Dịch vụ Hoa Xinh" data-nimg="1" decoding="async" height="90" loading="lazy" src="https://careerviet.vn/_next/image?url=https%3A%2F%2Fimages.careerviet.vn%2Femployer_folders%2Flot8%2F326898%2F124615292324134_443547111112026_8795892205442577213_n.jpg&amp;w=256&amp;q=75" srcset="/_next/image?url=https%3A%2F%2Fimages.careerviet.vn%2Femployer_folders%2Flot8%2F326898%2F124615292324134_443547111112026_8795892205442577213_n.jpg&amp;w=96&amp;q=75 1x, /_next/image?url=https%3A%2F%2Fimages.careerviet.vn%2Femployer_folders%2Flot8%2F326898%2F124615292324134_443547111112026_8795892205442577213_n.jpg&amp;w=256&amp;q=75

In [14]:
pattern_value_tag = re.compile(r'(?=.*job-detail)(?=.*info)(?=.*section-content-value)', re.IGNORECASE)

job_tags = job_list_box.find_all(
    "div", 
    class_ =re.compile(r'(?=.*job-item)', re.IGNORECASE), 
    id     =re.compile(r'(?=.*job-item)', re.IGNORECASE))
    
job_tags

[<div class="job-item" id="job-item-35C71953"><div class="figure"><div class="asset-wrapper"><div class="image"><a href="/vi/nha-tuyen-dung/cong-ty-tnhh-san-xuat-thuong-mai-dich-vu-hoa-xinh.35A9E5F2.html" target="_blank" title="Công ty TNHH Sản xuất Thương mại &amp; Dịch vụ Hoa Xinh"><img alt="Công ty TNHH Sản xuất Thương mại &amp; Dịch vụ Hoa Xinh" data-nimg="1" decoding="async" height="90" loading="lazy" src="https://careerviet.vn/_next/image?url=https%3A%2F%2Fimages.careerviet.vn%2Femployer_folders%2Flot8%2F326898%2F124615292324134_443547111112026_8795892205442577213_n.jpg&amp;w=256&amp;q=75" srcset="/_next/image?url=https%3A%2F%2Fimages.careerviet.vn%2Femployer_folders%2Flot8%2F326898%2F124615292324134_443547111112026_8795892205442577213_n.jpg&amp;w=96&amp;q=75 1x, /_next/image?url=https%3A%2F%2Fimages.careerviet.vn%2Femployer_folders%2Flot8%2F326898%2F124615292324134_443547111112026_8795892205442577213_n.jpg&amp;w=256&amp;q=75 2x" style="color: transparent;" width="90"/></a></div>

In [24]:
# Tìm thẻ <h2>, bên trong có thẻ <a>, trong <a> có thuộc tính class="job_link" và thuộc tính href.

job_tags = job_list_box.select('h2 a.job_link[href]')
job_tags

[<a class="job_link" data-id="35C71953" href="/vi/tim-viec-lam/tuyen-dung-thuc-tap-sinh-tuyen-dung.35C71953.html" rel="noopener noreferrer" target="_blank" title="Tuyển Dụng Thực Tập Sinh Tuyển Dụng">Tuyển Dụng Thực Tập Sinh Tuyển Dụng</a>,
 <a class="job_link" data-id="35C6FDFF" href="/vi/tim-viec-lam/chuyen-vien-phat-trien-corebank-du-an-nang-cap-corebanking.35C6FDFF.html" rel="noopener noreferrer" target="_blank" title="Chuyên viên phát triển CoreBank - Dự án nâng cấp CoreBanking">Chuyên viên phát triển CoreBank - Dự án nâng cấp CoreBanking</a>,
 <a class="job_link" data-id="35C6F679" href="/vi/tim-viec-lam/data-engineer.35C6F679.html" rel="noopener noreferrer" target="_blank" title="Data Engineer">Data Engineer</a>,
 <a class="job_link" data-id="35C6B757" href="/vi/tim-viec-lam/giam-doc-mo-hinh-rui-ro-digital-risk-modeling-head-tnex-4b035.35C6B757.html" rel="noopener noreferrer" target="_blank" title="Giám đốc Mô hình Rủi ro (Digital Risk Modeling Head) - TNEX - 4B035">Giám đốc Mô 

In [30]:
len(job_tags)

38

In [25]:
raw_link = job_tags[0].get('href')
raw_link

'/vi/tim-viec-lam/tuyen-dung-thuc-tap-sinh-tuyen-dung.35C71953.html'

In [26]:
clean_link = "https://careerviet.vn" + raw_link.split('?')[0]
clean_link

'https://careerviet.vn/vi/tim-viec-lam/tuyen-dung-thuc-tap-sinh-tuyen-dung.35C71953.html'

In [34]:
def get_links(soup):
    all_links = []
    try:
        # 1. Tìm container chứa danh sách công việc BẰNG "jobs-side-list" (đã dùng find all và chỉ ra 1 kết quả duy nhất, trộm vía)
        job_list_box = soup.find("div", class_ = "jobs-side-list", id="jobs-side-list-content")

        
        if job_list_box:
            # 2. Tìm tất cả các thẻ <a> chứa link job. 
            # Ở đây ta lấy thẻ <a> nằm trong <h2>, trong <a> có thuộc tính class="job_link" và thuộc tính href.
            a_tags = job_list_box.select('h2 a.job_link[href]')
            
            for a in a_tags:
                raw_link = a.get('href')
                
                if raw_link:
                    # 3. LÀM SẠCH LINK: Cắt chuỗi tại dấu '?' và chỉ lấy phần đầu tiên (index 0)
                    # Ví dụ: ".../2106718.html?ta_source=..." -> ".../2106718.html"
                    clean_link = "https://careerviet.vn" + raw_link.split('?')[0]
                    all_links.append(clean_link)
                    
    except Exception as e:
        logging.error(f"Lỗi khi lấy danh sách link: {e}")
        pass
        
    return all_links

In [35]:
# 1 trang sẽ MAX 50 JOB

job_links = get_links(soup)
job_links

['https://careerviet.vn/vi/tim-viec-lam/software-architect.35C6F3A3.html',
 'https://careerviet.vn/vi/tim-viec-lam/software-architect.35C6BE06.html',
 'https://careerviet.vn/vi/tim-viec-lam/software-architect.35C6AB02.html',
 'https://careerviet.vn/vi/tim-viec-lam/ai-software-engineer-python-go-c-c-backend-ai-agents-khoi-cong-nghe-thong-tin-holt-19.35C6C907.html',
 'https://careerviet.vn/vi/tim-viec-lam/agentic-engineer-python-go-c-c-backend-ai-agents-khoi-cong-nghe-thong-tin-holt-18.35C6C906.html',
 'https://careerviet.vn/vi/tim-viec-lam/ai-software-engineer-python-go-c-c-backend-ai-agents-khoi-cong-nghe-thong-tin-holt-19.35C6FC79.html',
 'https://careerviet.vn/vi/tim-viec-lam/senior-transport-solution-design-specialist.35C6D3FB.html',
 'https://careerviet.vn/vi/tim-viec-lam/senior-incentive-commission-specialist-sales-operations.35C72173.html',
 'https://careerviet.vn/vi/tim-viec-lam/nhan-vien-marketing.35C6E38C.html',
 'https://careerviet.vn/vi/tim-viec-lam/senior-risk-modelling-dev

In [36]:
len(job_links)

50

# 3. Con trỏ chuột sang trang kế tiếp
Output test với **`KEYWORD`** = `data analyst` là lấy đc `188 job` từ 4 page

In [56]:
KEYWORD = "data-analyst"
MAX_PAGES = 7 # Đặt số trang tối đa bạn muốn cào
all_job_links = [] # List tổng chứa tất cả các link thu thập được từ mọi trang

print(f"\n{'='*50}")
print(f"BẮT ĐẦU CÀO DANH SÁCH LINK CHO TỪ KHÓA: {KEYWORD.upper()}")
print(f"{'='*50}")


BẮT ĐẦU CÀO DANH SÁCH LINK CHO TỪ KHÓA: DATA-ANALYST


In [53]:
# BASE_URL_TEMPLATE = "https://careerviet.vn/viec-lam/{keyword}-k-sortdv-trang-{page}-vi.html"

# Vòng lặp chạy từ trang 1 đến MAX_PAGES
for page in range(1, MAX_PAGES + 1):
    # 1. Tạo URL cho trang hiện tại
    url = f'https://careerviet.vn/viec-lam/{KEYWORD}-k-sortdv-trang-{page}-vi.html'
    print(f"\n---> Đang truy cập Trang {page}: {url}")


    # 2. Phải cho trình duyệt chạy từng link
    driver.get(url)
    time.sleep(random.uniform(3, 6)) # Đợi 3 giây để trang web tải hoàn toàn


    # 3. Cuộn xuống để load toàn bộ job
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height



    # 4. Lấy mã nguồn HTML đưa cho BeautifulSoup
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')    
    
    job_list_box = soup.find("div", class_ = "jobs-side-list", id="jobs-side-list-content")
    

    # 5. KIỂM TRA ĐIỀU KIỆN DỪNG (BREAK)  
    if  not job_list_box:
        print(f"[THÔNG BÁO] Không tìm thấy khung danh sách job ở trang {page}. Dừng cào!")
        break # Thoát khỏi vòng lặp for hoàn toàn

    
    job_tags = job_list_box.select('h2 a.job_link[href]')
    if len(job_tags) > 0:
        print(f"\n---> Trang {page} có {len(job_tags)} jobs")
    
    else:
        print(f"[THÔNG BÁO] Không tìm thấy khung danh sách job ở trang {page}. Dừng cào!")
        break # Thoát khỏi vòng lặp for hoàn toàn

        


---> Đang truy cập Trang 1: https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-1-vi.html

---> Trang 1 có 50 jobs

---> Đang truy cập Trang 2: https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-2-vi.html

---> Trang 2 có 50 jobs

---> Đang truy cập Trang 3: https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-3-vi.html

---> Trang 3 có 50 jobs

---> Đang truy cập Trang 4: https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-4-vi.html

---> Trang 4 có 38 jobs

---> Đang truy cập Trang 5: https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-5-vi.html
[THÔNG BÁO] Không tìm thấy khung danh sách job ở trang 5. Dừng cào!


In [57]:
# Vòng lặp đầy đủ, và chính thức
# Vòng lặp chạy từ trang 1 đến MAX_PAGES
for page in range(1, MAX_PAGES + 1):
    # 1. Tạo URL cho trang hiện tại
    url = f'https://careerviet.vn/viec-lam/{KEYWORD}-k-sortdv-trang-{page}-vi.html'
    print(f"\n---> vào page {page}: {url}")
    

    # 2. Phải cho trình duyệt chạy từng link
    driver.get(url)
    time.sleep(random.uniform(3, 6)) # Đợi 3 giây để trang web tải hoàn toàn



    # 3. Cuộn xuống cuối trang để load toàn bộ job (Lazy load)
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    # 4. Lấy mã nguồn HTML đưa cho BeautifulSoup
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # 5. Tìm container chứa danh sách công việc
    job_list_box = soup.find("div", class_ = "jobs-side-list", id="jobs-side-list-content")

    # 6. KIỂM TRA ĐIỀU KIỆN DỪNG (BREAK)
    if not job_list_box:
        print(f"[THÔNG BÁO] Page {page} ko có job nào. Đã hết dữ liệu. Dừng cào!")
        break # Thoát khỏi vòng lặp for hoàn toàn

    # 6. Tìm TẤT CẢ các thẻ <h2> chứa link job
    job_tags = job_list_box.select('h2 a.job_link[href]')
    
    # Nếu khung jobList tồn tại nhưng bên trong không có job nào (trang trống)
    if len(job_tags) == 0:
        print(f"[THÔNG BÁO] Page {page} ko có job nào. Đã hết dữ liệu. Dừng cào!")
        break


    # 7. Lặp qua từng thẻ <h2> để lấy link và đưa vào list tổng
    page_links_count = 0
    for tag in job_tags:
        link = tag.get('href')
        if link:
            clean_link = "https://careerviet.vn" + raw_link.split('?')[0]
            all_job_links.append(clean_link)
            page_links_count += 1
            
    print(f"[*] Save {page_links_count} link page {page} vào list_jobs.")


# ==========================================
# (Tùy chọn) In thử 5 link đầu tiên ra xem
for i, url in enumerate(all_job_links, 1):
    print(f"{i}. {url}")


---> vào page 1: https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-1-vi.html
[*] Save 50 link page 1 vào list_jobs.

---> vào page 2: https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-2-vi.html
[*] Save 50 link page 2 vào list_jobs.

---> vào page 3: https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-3-vi.html
[*] Save 50 link page 3 vào list_jobs.

---> vào page 4: https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-4-vi.html
[*] Save 38 link page 4 vào list_jobs.

---> vào page 5: https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-5-vi.html
[THÔNG BÁO] Page 5 ko có job nào. Đã hết dữ liệu. Dừng cào!
1. https://careerviet.vn/vi/tim-viec-lam/tuyen-dung-thuc-tap-sinh-tuyen-dung.35C71953.html
2. https://careerviet.vn/vi/tim-viec-lam/tuyen-dung-thuc-tap-sinh-tuyen-dung.35C71953.html
3. https://careerviet.vn/vi/tim-viec-lam/tuyen-dung-thuc-tap-sinh-tuyen-dung.35C71953.html
4. https://careerviet.vn/vi/tim-viec-lam/tuyen-dung-thuc-tap-sinh-tuyen-dung

# 4. Bắt đầu lấy data sản phẩm nè

1.	`job_url`: Link chi tiết của job đó.
2.	`job_title`: Tiêu đề công việc.
3.	`company_name`: Tên công ty.

4.	`salary`: Mức lương (thỏa thuận OR rank min - max).
5.	`location`: Địa điểm làm việc.
6.	`years_experience`: kinh nghiệm

7.	`job_deadline`: (Hạn nộp hồ sơ)
8.	`date_post`: Tính từ ngày dealine trừ đi 1 tháng

9.	`level_employee`: cấp bậc (Nhân viên)
10.	`job_type`: Hình thức làm việc ('nhân viên chính thức', t đoán chắc fulltime)

11.	`job_description`: Mô tả công việc.
12.	`requirements`: Yêu cầu công việc (Kinh nghiệm, kỹ năng).
13.	`benefits`: Phúc lợi (Lý do tại sao bạn sẽ yêu thích làm việc tại đây).
14. `other_info`: các thông tin về bằng cấp, lương, độ tuổi

In [6]:
# https://careerviet.vn/vi/tim-viec-lam/ha-noi-business-intelligence-executive-data-analyst-executive.35C6913E.html

job_link = 'https://careerviet.vn/vi/tim-viec-lam/ha-noi-business-intelligence-executive-data-analyst-executive.35C6913E.html'
job_link = job_link.split('?')[0]
job_link

'https://careerviet.vn/vi/tim-viec-lam/ha-noi-business-intelligence-executive-data-analyst-executive.35C6913E.html'

In [7]:

# Mở trang job chi tiết
driver.get(job_link)
time.sleep(random.uniform(3, 6)) # Đợi 3 giây để trang web tải hoàn toàn

# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

In [8]:
# Lấy mã nguồn HTML của trang web
new_html = driver.page_source
new_soup = BeautifulSoup(new_html, 'html.parser')

In [ ]:
new_soup

## 4.1 	Title Job

In [63]:
title_tag = new_soup.find('h1', class_='title')
title_tag

[<h1 class="title">[Hà Nội] Business Intelligence Executive (Data Analyst Executive)</h1>]

In [62]:
title_tag.get_text(separator=' ', strip=True)

'[Hà Nội] Business Intelligence Executive (Data Analyst Executive)'

In [64]:
# function 1: lấy tiêu đề job

def get_title(soup):
    try:
        # 1. Tìm thẻ h1 class = title, là tag duy nhất
        title_tag = new_soup.find('h1', class_='title')
            
        if title_tag:
            raw_title = title_tag.get_text(separator=' ', strip=True)
            return raw_title
                
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy title: {e}")
        return None

In [65]:
# test function: lấy tiêu đề job
title = get_title(new_soup)
title

'[Hà Nội] Business Intelligence Executive (Data Analyst Executive)'

## 4.2  Company name

In [67]:
# <a> class = employer job-company-name
company_tag = new_soup.find('a', class_=re.compile(r'(?=.*job-company-name)', re.IGNORECASE))
company_tag

<a class="employer job-company-name" href="/vi/nha-tuyen-dung/cong-ty-co-phan-diana-unicharm.35A6E11B.html" rel="noreferrer" target="_blank">Công Ty Cổ Phần Diana Unicharm</a>

In [68]:
company_name = company_tag.get_text(strip=True)
company_name

'Công Ty Cổ Phần Diana Unicharm'

In [69]:
# function 6: lấy tên cty

def get_company(soup):
    try:
        # <a> class = employer job-company-name
        company_tag = new_soup.find('a', class_=re.compile(r'(?=.*job-company-name)', re.IGNORECASE))
        
        if company_tag:
            return company_tag.get_text(strip=True)

        return None
    except Exception:
        return None


company_name = get_company(new_soup)
company_name

'Công Ty Cổ Phần Diana Unicharm'

##      Nguyên cái khối thông tin job

In [9]:
# job-detail-content
box_info = new_soup.find('section', class_='job-detail-content')
box_info

<section class="job-detail-content"><div class="bg-blue"><div class="row"><div class="col-lg-4 col-md-4 col-sm-6 item-blue"><div class="detail-box"><div class="map"><strong><em class="mdi mdi-map-marker"></em>Địa điểm</strong><p><a href="/viec-lam/ha-noi-l4-vi.html" title="Hà Nội">Hà Nội</a></p><a aria-controls="radix-«r11»" aria-expanded="false" aria-haspopup="dialog" data-state="closed" role="button" tabindex="0" type="button"><img alt="Hà Nội" data-nimg="1" decoding="async" height="122" loading="lazy" src="https://assets.careerviet.vn/careerviet/_next/static/media/icon-map-has-text.f225301f.svg" style="color: transparent;" width="293"/></a></div></div><div class="detail-box has-background" style="display: none;"><ul><li><strong><em class="mdi mdi-update"> </em>Ngày cập nhật</strong><p>27/03/2026</p></li><li><strong><em class="mdi mdi-briefcase"></em>Ngành nghề</strong><p><a href="/viec-lam/cntt-phan-mem-c1-vi.html" title="CNTT - Phần mềm">CNTT - Phần mềm</a>, <a href="/viec-lam/thuo

### 4.3. Location

In [79]:
# ra kết quả duy nhất, trộm vía
location_box = box_info.find('div', class_='map')
location_box

<div class="map"><strong><em class="mdi mdi-map-marker"></em>Địa điểm</strong><p><a href="/viec-lam/ha-noi-l4-vi.html" title="Hà Nội">Hà Nội</a></p><a aria-controls="radix-«r11»" aria-expanded="false" aria-haspopup="dialog" data-state="closed" role="button" tabindex="0" type="button"><img alt="Hà Nội" data-nimg="1" decoding="async" height="122" loading="lazy" src="https://assets.careerviet.vn/careerviet/_next/static/media/icon-map-has-text.f225301f.svg" style="color: transparent;" width="293"/></a></div>

In [84]:
# ra kết quả duy nhất, trộm vía
location_tag = location_box.find('p')
location_tag

<p><a href="/viec-lam/ha-noi-l4-vi.html" title="Hà Nội">Hà Nội</a></p>

In [85]:
location = location_tag.get_text(strip=True)
location

'Hà Nội'

In [12]:
# function 3: lấy địa điểm việc làm

def get_location(box_info):
    try:
        location_box = box_info.find('div', class_='map')
        
        if location_box:
            location_tag = location_box.find('p')

            if location_tag:
                return location_tag.get_text(strip=True)
        return None
    except Exception:
        return None


location = get_location(box_info)
location

'Hà Nội'

### 4.4. posted_date

In [25]:
# 1. Tìm thẻ strong có chứa chữ "Ngày cập nhật"
posted_date_title = box_info.find(
    lambda tag: tag.name == 'strong' 
    and    'Ngày cập nhật' in tag.get_text(strip=True)
)
posted_date_title

<strong><em class="mdi mdi-update"> </em>Ngày cập nhật</strong>

In [20]:
# 2. Tìm thẻ <p> là anh em liền kề của thẻ <strong>
posted_date_tag = posted_date_title.find_next_sibling('p')
posted_date_tag

<p>27/03/2026</p>

In [21]:
posted_date = posted_date_tag.get_text(strip=True)
posted_date

'27/03/2026'

In [22]:
# function: lấy ngày đăng của job

def get_posted_date(box_info):
    try:
        # 1. Tìm thẻ strong có chứa chữ "Ngày cập nhật"
        posted_date_title = box_info.find(
            lambda tag: tag.name == 'strong' and 'Ngày cập nhật' in tag.get_text(strip=True)
        )
        
        if posted_date_title:
            posted_date_tag = posted_date_title.find_next_sibling('p')

            if posted_date_tag:
                return posted_date_tag.get_text(strip=True)
        return None
    except Exception:
        return None


posted_date = get_posted_date(box_info)
posted_date

'27/03/2026'

## Cách khác là Dictionary: 
Dùng cách này ổn hơn
### 4.4. posted_date
### 4.5. deadline_date
### 4.6. category
### 4.7. work_type
### 4.8. salary
### 4.9. experience
### 4.10. level_employee

In [32]:
def get_careerviet_job_details(box_info):
    job_info = {}
    try:
        # Tìm tất cả các thẻ <li> trong khối detail-box
        li_tags = box_info.find_all('li')
        
        for li in li_tags:
            strong_tag = li.find('strong')
            p_tag = li.find('p')
            
            if strong_tag and p_tag:
                # Lấy Key (Xóa khoảng trắng thừa do cái icon <em> để lại)
                key = strong_tag.get_text(strip=True)
                
                # Lấy Value (Dùng separator để các ngành nghề không bị dính liền nhau)
                value = p_tag.get_text(separator=" ", strip=True)
                
                job_info[key] = value
                
        return job_info
    except Exception as e:
        print(f"Lỗi: {e}")
        return job_info

# ================= ÁP DỤNG =================
job_info = get_careerviet_job_details(box_info)
print(job_info)

# Kết quả sẽ ra một Dictionary cực đẹp:
# {
#   'Ngày cập nhật': '27/03/2026', 
#   'Ngành nghề': 'CNTT - Phần mềm, Thương mại điện tử, Ngành khác', 
#   'Hình thức': 'Nhân viên chính thức', 
#   'Lương': '28 Tr - 40 Tr VND', 
#   'Kinh nghiệm': '3 - 5 Năm', 
#   'Cấp bậc': 'Nhân viên', 
#   'Hết hạn nộp': '25/04/2026'
# }

# Muốn lấy ngày cập nhật? Quá dễ:
posted_date = job_info.get('Ngày cập nhật')
deadline_date = job_info.get('Hết hạn nộp')
category = job_info.get('Ngành nghề')
work_type = job_info.get('Hình thức')
salary = job_info.get('Lương')
experience = job_info.get('Kinh nghiệm')
level_employee = job_info.get('Cấp bậc')


{'Ngày cập nhật': '27/03/2026', 'Ngành nghề': 'CNTT - Phần mềm , Thương mại điện tử , Ngành khác', 'Hình thức': 'Nhân viên chính thức', 'Lương': '28 Tr - 40 Tr VND', 'Kinh nghiệm': '3 - 5 Năm', 'Cấp bậc': 'Nhân viên', 'Hết hạn nộp': '25/04/2026'}


In [33]:
posted_date

'27/03/2026'

In [34]:
deadline_date

'25/04/2026'

In [39]:
category

'CNTT - Phần mềm , Thương mại điện tử , Ngành khác'

In [38]:
work_type

'Nhân viên chính thức'

In [37]:
salary

'28 Tr - 40 Tr VND'

In [36]:
experience

'3 - 5 Năm'

In [35]:
level_employee

'Nhân viên'

## Nguyên Cái Khối Dưới

### 4.11       education_level

In [42]:
# 1. Tìm cái box "Thông tin khác" bên dưới cùng, chỉ ra 1 kết quả duy nhất
other_box_title = box_info.find('h3', class_="detail-title", string='Thông tin khác')
other_box_title

<h3 class="detail-title">Thông tin khác</h3>

In [43]:
other_box = other_box_title.parent
other_box

<div class="detail-row"><h3 class="detail-title">Thông tin khác</h3><div class="content_fck"><ul><li>Bằng cấp: Đại học</li><li>Độ tuổi: Không giới hạn tuổi</li><li>Lương: 28 Tr - 40 Tr VND</li></ul></div></div>

In [48]:
# 2. Lấy tag bằng cấp
edu_tag = other_box.find('li', string=re.compile(r'Bằng cấp', re.IGNORECASE))
edu_tag

<li>Bằng cấp: Đại học</li>

In [49]:
# 3. Lấy text
raw_text = edu_tag.get_text(strip=True) # VD: "Bằng cấp: Đại học"
raw_text

'Bằng cấp: Đại học'

In [50]:
# 4. Xử lý text
if ':' in raw_text:
    # Lấy phần tử thứ 2 (index 1) sau dấu ':' và xóa khoảng trắng
    education_level = raw_text.split(':', 1)[1].strip() 


education_level

'Đại học'

In [51]:


def get_education_level(box_info):
    """
    Tìm và lấy trực tiếp thông tin "Bằng cấp" trong khối 'Thông tin khác'.
    """
    try:
        # 1. Tìm cái box "Thông tin khác" bên dưới cùng, chỉ ra 1 kết quả duy nhất
        other_box_title = box_info.find('h3', class_="detail-title", string='Thông tin khác')

        if other_box_title:
            other_box = other_box_title.parent

            
            # 2. Lấy tag bằng cấp
            edu_tag = other_box.find('li', string=re.compile(r'Bằng cấp', re.IGNORECASE))
            

            if edu_tag:
                raw_text = edu_tag.get_text(strip=True) # VD: "Bằng cấp: Đại học"
                
                # 3. Cắt chuỗi lấy phần Value
                if ':' in raw_text:
                    # Lấy phần tử thứ 2 (index 1) sau dấu ':' và xóa khoảng trắng
                    return raw_text.split(':', 1)[1].strip() 
                    
        return None

    except Exception as e:
        logging.error(f"Lỗi khi lấy Bằng cấp: {e}")
        return None

# ================= ÁP DỤNG =================
education_level = get_education_level(box_info)
education_level

'Đại học'

### 4.12      MÔ TẢ CÔNG VIỆC
### 4.13      YÊU CẦU CÔNG VIỆC
### 4.14      PHÚC LỢI

In [57]:
# 1. Tìm thẻ h2 chứa từ khóa (Mỏ neo) - Không phân biệt hoa thường
# t phát hiện ra 1 tác dụng của Regex là nó ko phân biệt viết HOA - thường
h2_tag = box_info.find(
    'h2', 
    class_='detail-title', 
    string=re.compile('Mô tả công việc', re.IGNORECASE))
h2_tag

<h2 class="detail-title">Mô tả Công việc</h2>

In [58]:
# 2. Lấy tất cả các thẻ nằm SAU thẻ h3 này (p, ul, li...)
h2_tag.find_next_siblings()

[<div><p>• Phân tích, xử lý Dữ liệu &amp; BI</p>
 <p><em>Data Analytics &amp; BI, Data Processing</em></p>
 <p>• Thu thập, làm sạch dữ liệu B2C/C2C; sử dụng công cụ crawling và cơ chế chống chặn.</p>
 <p><em>Collect/clean B2C/C2C data; utilize crawling &amp; anti-blocking tools.</em></p>
 <p>• Kiến trúc &amp; Báo cáo</p>
 <p><em>Architecture &amp; Reporting</em></p>
 <p>• Xây dựng kho dữ liệu (DW/Mart) và dashboard tự động (Tableau, Power BI, Looker).</p>
 <p><em>Develop Data Warehouses/Marts and automated BI dashboards.</em></p>
 <p>• Theo dõi Hiệu suất</p>
 <p><em>Performance Monitoring</em></p>
 <p>• Quản lý KPI, A/B testing và đo lường hiệu quả chiến dịch TMĐT.</p>
 <p><em>Manage KPIs, A/B testing, and E-commerce performance tracking</em>.</p>
 <p>• Trực quan hóa &amp; Insights</p>
 <p><em>Visualization &amp; Insights</em></p>
 <p>• Trình bày dữ liệu phức tạp thành thông tin dễ hiểu, hỗ trợ ra quyết định chiến lược.</p>
 <p><em>Transform complex data into actionable insights for st

In [59]:
content_parts = []

for sibling in h2_tag.find_next_siblings():
    # Lấy text, dùng dấu cách để các chữ không bị dính vào nhau
    text = sibling.get_text(separator=' ', strip=True)
    if text:
        content_parts.append(text)

content_parts        

['• Phân tích, xử lý Dữ liệu & BI Data Analytics & BI, Data Processing • Thu thập, làm sạch dữ liệu B2C/C2C; sử dụng công cụ crawling và cơ chế chống chặn. Collect/clean B2C/C2C data; utilize crawling & anti-blocking tools. • Kiến trúc & Báo cáo Architecture & Reporting • Xây dựng kho dữ liệu (DW/Mart) và dashboard tự động (Tableau, Power BI, Looker). Develop Data Warehouses/Marts and automated BI dashboards. • Theo dõi Hiệu suất Performance Monitoring • Quản lý KPI, A/B testing và đo lường hiệu quả chiến dịch TMĐT. Manage KPIs, A/B testing, and E-commerce performance tracking . • Trực quan hóa & Insights Visualization & Insights • Trình bày dữ liệu phức tạp thành thông tin dễ hiểu, hỗ trợ ra quyết định chiến lược. Transform complex data into actionable insights for strategic decision-making. • Phối hợp liên phòng ban và phân tích chiến lược/giá cả của đối thủ cạnh tranh. Collaborate with stakeholders and analyze strategies • Đảm bảo tính tuân thủ dữ liệu; đào tạo văn hóa dữ liệu và cả

In [60]:
# 3. Nối tất cả lại thành 1 đoạn văn duy nhất
# Đã check, và nó cào toàn bộ nội dung phần Mô tả thật
raw_text = ' '.join(content_parts)
raw_text

'• Phân tích, xử lý Dữ liệu & BI Data Analytics & BI, Data Processing • Thu thập, làm sạch dữ liệu B2C/C2C; sử dụng công cụ crawling và cơ chế chống chặn. Collect/clean B2C/C2C data; utilize crawling & anti-blocking tools. • Kiến trúc & Báo cáo Architecture & Reporting • Xây dựng kho dữ liệu (DW/Mart) và dashboard tự động (Tableau, Power BI, Looker). Develop Data Warehouses/Marts and automated BI dashboards. • Theo dõi Hiệu suất Performance Monitoring • Quản lý KPI, A/B testing và đo lường hiệu quả chiến dịch TMĐT. Manage KPIs, A/B testing, and E-commerce performance tracking . • Trực quan hóa & Insights Visualization & Insights • Trình bày dữ liệu phức tạp thành thông tin dễ hiểu, hỗ trợ ra quyết định chiến lược. Transform complex data into actionable insights for strategic decision-making. • Phối hợp liên phòng ban và phân tích chiến lược/giá cả của đối thủ cạnh tranh. Collaborate with stakeholders and analyze strategies • Đảm bảo tính tuân thủ dữ liệu; đào tạo văn hóa dữ liệu và cải

In [62]:

    
# Xóa các ký tự bullet point thường gặp (Thêm dấu '-' vì nhiều HR dùng gạch đầu dòng)
clean_text = re.sub(r'[•\*●\-\+\−]', '', raw_text)

# Tuyệt chiêu gom khoảng trắng: Xóa sạch \n, \t, \xa0 và các dấu cách thừa
final_text = ' '.join(clean_text.split())

# Chống lỗi công thức khi xuất ra file CSV/Excel
if final_text.startswith(('-', '=', '+', '@')):
    final_text = "'" + final_text


final_text

'Phân tích, xử lý Dữ liệu & BI Data Analytics & BI, Data Processing Thu thập, làm sạch dữ liệu B2C/C2C; sử dụng công cụ crawling và cơ chế chống chặn. Collect/clean B2C/C2C data; utilize crawling & antiblocking tools. Kiến trúc & Báo cáo Architecture & Reporting Xây dựng kho dữ liệu (DW/Mart) và dashboard tự động (Tableau, Power BI, Looker). Develop Data Warehouses/Marts and automated BI dashboards. Theo dõi Hiệu suất Performance Monitoring Quản lý KPI, A/B testing và đo lường hiệu quả chiến dịch TMĐT. Manage KPIs, A/B testing, and Ecommerce performance tracking . Trực quan hóa & Insights Visualization & Insights Trình bày dữ liệu phức tạp thành thông tin dễ hiểu, hỗ trợ ra quyết định chiến lược. Transform complex data into actionable insights for strategic decisionmaking. Phối hợp liên phòng ban và phân tích chiến lược/giá cả của đối thủ cạnh tranh. Collaborate with stakeholders and analyze strategies Đảm bảo tính tuân thủ dữ liệu; đào tạo văn hóa dữ liệu và cải tiến hệ thống. Ensure 

In [72]:
# Cào cùng lúc 3 mục, vì ở trong cùng 1 box

def extract_paragraph_content(box_info, heading_keyword):
    """
    Hàm đa năng: Tìm thẻ h2 chứa keyword, sau đó lấy toàn bộ text 
    của các thẻ anh em (siblings) nằm dưới nó trong cùng 1 div.
    """
    try:
        # 1. thẻ title của box:
        pattern_string = re.compile(rf'{heading_keyword}', re.IGNORECASE)
        h2_tag = box_info.find('h2', class_='detail-title', string=pattern_string)


        # 2. Tìm thẻ cha là BOX chứa cái thông tin đó
        if h2_tag:

            # 2. Lấy nguyên text
            content_parts = []

            for sibling in h2_tag.find_next_siblings():
                # Lấy text, dùng '. ' làm dải phân cách cho các thẻ con (như <li>, <br>)
                text = sibling.get_text(separator='. ', strip=True)
                if text:
                    content_parts.append(text)



            # 3. Nối tất cả lại thành 1 đoạn văn duy nhất
            # Đã check, và nó cào toàn bộ nội dung phần Mô tả thật
            raw_text = ' '.join(content_parts)


            # 4. Làm sạch text
            # Xóa các ký tự bullet point thường gặp (Thêm dấu '-' vì nhiều HR dùng gạch đầu dòng)
            clean_text = re.sub(r'[•\*●\-\+\−]', '', raw_text)
            
            # Tuyệt chiêu gom khoảng trắng: Xóa sạch \n, \t, \xa0 và các dấu cách thừa
            final_text = ' '.join(clean_text.split())
            
            # Chống lỗi công thức khi xuất ra file CSV/Excel
            if final_text.startswith(('-', '=', '+', '@')):
                final_text = "'" + final_text
        
            return final_text
            
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy {heading_keyword}: {e}")
        return None



# ==========================================
# ÁP DỤNG VÀO 3 HÀM CỦA BẠN
# ==========================================

def get_job_description(box_info):
    return extract_paragraph_content(box_info, "Mô tả công việc")

def get_job_requirements(box_info):
    return extract_paragraph_content(box_info, "Yêu Cầu Công Việc")

def get_benefits(box_info):
    return extract_paragraph_content(box_info, "Phúc lợi")

In [73]:
description = get_job_description(box_info)
description

'Phân tích, xử lý Dữ liệu & BI. Data Analytics & BI, Data Processing. Thu thập, làm sạch dữ liệu B2C/C2C; sử dụng công cụ crawling và cơ chế chống chặn.. Collect/clean B2C/C2C data; utilize crawling & antiblocking tools.. Kiến trúc & Báo cáo. Architecture & Reporting. Xây dựng kho dữ liệu (DW/Mart) và dashboard tự động (Tableau, Power BI, Looker).. Develop Data Warehouses/Marts and automated BI dashboards.. Theo dõi Hiệu suất. Performance Monitoring. Quản lý KPI, A/B testing và đo lường hiệu quả chiến dịch TMĐT.. Manage KPIs, A/B testing, and Ecommerce performance tracking. .. Trực quan hóa & Insights. Visualization & Insights. Trình bày dữ liệu phức tạp thành thông tin dễ hiểu, hỗ trợ ra quyết định chiến lược.. Transform complex data into actionable insights for strategic decisionmaking.. Phối hợp liên phòng ban và phân tích chiến lược/giá cả của đối thủ cạnh tranh.. Collaborate with stakeholders and analyze strategies. Đảm bảo tính tuân thủ dữ liệu; đào tạo văn hóa dữ liệu và cải tiế

In [74]:
requirements = get_job_requirements(box_info)
requirements

'Tốt nghiệp Đại học các ngành Kinh tế, Ngoại thương, TM, QTKD hoặc Tài chính,.... Bachelor’s degree in Economics, Foreign Trade, Business, or Finance,.... Từ 3 năm kinh nghiệm Chuyên viên phân tích kinh doanh tại vị trí tương đương hoặc Kế hoạch kinh doanh/Trade trong ngành FMCG. Min. 3 years in a similar role or Sales/Trade Planning within FMCG. Ưu tiên: Có 23 năm kinh nghiệm trong lĩnh vực kinh doanh trực tuyến (Online Business).. Preferred: 23 years of experience in online business.. Thành thạo các công cụ phân tích KD như Tableau, Power BI, hoặc Looker Studio.. Experience with BI tools such as Tableau, Power BI, or Looker Studio.. Strong analytical thinking and problemsolving skills.. Tư duy phản biện và giải quyết vấn đề tốt.. Attention to detail and commitment to data accuracy.. Chú trọng chi tiết và tính chính xác của dữ liệu.. Có kiến thức về các chỉ số TMĐT, Marketing và Bán hàng.. Understanding of ecommerce operations, marketing, and sales metrics.. English communication skil

In [75]:
benefits = get_benefits(box_info)
benefits

'Laptop. Chế độ bảo hiểm. Du Lịch. Du lịch nước ngoài. Chế độ thưởng. Chăm sóc sức khỏe. Đào tạo. Tăng lương. Công tác phí. Nghỉ phép năm. CLB thể thao'